In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time

# -------- MediaPipe --------
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    model_complexity=0,  # FAST
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

mp_selfie = mp.solutions.selfie_segmentation
segment = mp_selfie.SelfieSegmentation(model_selection=0)  # FAST

# -------- Camera --------
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

background = None
invisible = False

# -------- FPS Control --------
prev_time = 0

# -------- Gesture Stability --------
counter = 0
threshold_frames = 5   # avoid flicker

print("✋ Hand UP = Invisible | 👇 Hand DOWN = Visible")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)

    # -------- Background Capture --------
    if background is None:
        background = frame.copy()
        cv2.putText(frame, "Capturing Background...",
                    (20,40), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0,255,255), 2)
        cv2.imshow("Invisibility System", frame)
        cv2.waitKey(100)
        continue

    # -------- Resize (MAJOR SPEED BOOST) --------
    small = cv2.resize(frame, (320, 240))
    rgb_small = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)

    # -------- Hand Detection --------
    result = hands.process(rgb_small)

    detected = False

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            index_y = hand_landmarks.landmark[8].y
            wrist_y = hand_landmarks.landmark[0].y

            if index_y < wrist_y:
                detected = True

    # -------- Stability Logic --------
    if detected:
        counter += 1
    else:
        counter -= 1

    counter = max(0, min(counter, threshold_frames))

    if counter == threshold_frames:
        invisible = True
    elif counter == 0:
        invisible = False

    # -------- Invisibility --------
    if invisible:
        result_seg = segment.process(rgb_small)
        mask = result_seg.segmentation_mask

        mask = cv2.resize(mask, (frame.shape[1], frame.shape[0]))
        condition = mask > 0.6

        final = np.where(condition[:, :, None], background, frame)

        cv2.putText(final, "INVISIBLE MODE",
                    (20,40), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0,0,255), 2)
    else:
        final = frame
        cv2.putText(final, "VISIBLE MODE",
                    (20,40), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0,255,0), 2)

    # -------- FPS Display --------
    curr_time = time.time()
    fps = int(1 / (curr_time - prev_time + 0.0001))
    prev_time = curr_time

    cv2.putText(final, f"FPS: {fps}",
                (20,80), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (255,255,0), 2)

    cv2.imshow("Invisibility System", final)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()